# Frozen RoBERTa + MLP (2 classes)

**Hypothesis (Further Work item 1):** on frozen RoBERTa `<s>` embeddings, an MLP isn't much better than logistic regression. LR is a network with 0 hidden layers, so the MLP only adds nonlinear decision boundaries.

Same code as `Frozen Roberta MLP.ipynb`, but on A2's original binary task: `rec.sport.baseball` vs. `rec.sport.hockey`. Both classifiers get the exact same embeddings and training data, so the only difference is the classifier.

**Predict before running:** how many points will the MLP gain over LR?

## Load dataset (copied from A2)

In [1]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import fetch_20newsgroups

newsgroups_train = fetch_20newsgroups(
    subset='train', remove=('headers', 'footers', 'quotes'),
    categories=['rec.sport.baseball','rec.sport.hockey'])

newsgroups_test = fetch_20newsgroups(
    subset='test', remove=('headers', 'footers', 'quotes'),
    categories=['rec.sport.baseball','rec.sport.hockey'])

# Remove empty documents
for s, name in zip([newsgroups_train, newsgroups_test], ["train", "test"]):
  empty_indices = {i for i, doc in enumerate(s.data) if len(doc) == 0}
  orig_len = len(s.data)
  for k in ['data', 'filenames', 'target']:
    s[k] = [s[k][i] for i in range(orig_len) if i not in empty_indices]
  print(f"Removed {len(empty_indices)} empty documents from the {name} set. Before: {orig_len}. After: {len(s.data)}.")

y_train = np.array(newsgroups_train.target)
y_test = np.array(newsgroups_test.target)
n_classes = len(newsgroups_train.target_names)

Removed 26 empty documents from the train set. Before: 1197. After: 1171.
Removed 19 empty documents from the test set. Before: 796. After: 777.


## RoBERTa `<s>` embeddings

This is the same idea as A2's `RobertaVectorizer`: run each message through frozen `roberta-base` and keep the final-layer vector at position 0 (`<s>`). Two things are different:
- It calls Hugging Face directly and runs 32 messages per forward pass, so it takes a minute or two instead of one message at a time through spaCy. The result is saved to `roberta_cls_sports.npz`, so later runs (and the hyperparameter search) skip this cell's slow part.
- `<s>` sees the first 512 tokens, whereas A2's spaCy pipeline only embeds the first 128. So the LR number below won't exactly match A2's 91.25%.

In [2]:
import os
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
CACHE = "roberta_cls_sports.npz"

## X is a list of strings (messages). Returns a np 2D array of shape (len(X), 768):
## the <s> embedding of each message, like RobertaVectorizer.transform in A2
def roberta_embed(X, tokenizer, roberta, batch_size=32):
    vecs = []
    for i in tqdm(range(0, len(X), batch_size)):
        batch = tokenizer(X[i:i + batch_size], padding=True, truncation=True, max_length=512,
                          return_tensors="pt").to(device)
        with torch.no_grad():  # frozen: no gradients needed
            out = roberta(**batch)
        # [:, 0] = token 0 (<s>) of every message in the batch
        vecs.append(out.last_hidden_state[:, 0].cpu().numpy())
    return np.concatenate(vecs)

if os.path.exists(CACHE):
    saved = np.load(CACHE)
    X_train, X_test = saved["X_train"], saved["X_test"]
else:
    tokenizer = AutoTokenizer.from_pretrained("roberta-base")
    # add_pooling_layer=False skips the pooler: roberta-base has no trained weights for it
    # (that's the "newly initialized" warning from A2), and we only need last_hidden_state
    roberta = AutoModel.from_pretrained("roberta-base", add_pooling_layer=False).to(device).eval()
    X_train = roberta_embed(newsgroups_train.data, tokenizer, roberta)
    X_test = roberta_embed(newsgroups_test.data, tokenizer, roberta)
    np.savez(CACHE, X_train=X_train, X_test=X_test)

print(X_train.shape, X_test.shape)

/home/tonyxdsu/venvs/cs423/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(1171, 768) (777, 768)


## Baseline: logistic regression

First, hold out 10% of train as a dev set. The MLP needs it for early stopping; the test set is only used at the very end. LR trains on the same 90%, so the comparison stays fair.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import metrics

X_tr, X_dev, y_tr, y_dev = train_test_split(
    X_train, y_train, test_size=0.1, stratify=y_train, random_state=423)

logreg = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
lr_acc = metrics.accuracy_score(y_test, logreg.predict(X_test))
print(f"LR test accuracy: {lr_acc*100:.2f}")

LR test accuracy: 91.38


## MLP

768 → 1024 → 256 → 2. The first hidden layer is wide and has dropout 0.2; the second is narrower, with no dropout.
- **ReLU**: the standard hidden-layer activation (J&M Ch 6).
- **No softmax at the end**: `CrossEntropyLoss` applies it to the logits, as in Lecture 4's neural LM.
- **AdamW, lr 1e-3**: the usual starting learning rate for a small MLP. AdamW is Adam with weight decay (default 0.01), which pulls the weights toward 0 a little at every step, like LR's built-in regularization.

In [4]:
torch.manual_seed(423)

mlp = nn.Sequential(
    nn.Linear(768, 1024), nn.ReLU(), nn.Dropout(0.2),  # hidden layer 1: wide, with dropout
    nn.Linear(1024, 256), nn.ReLU(),                   # hidden layer 2: narrower, no dropout
    nn.Linear(256, n_classes),                         # one logit per class
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(mlp.parameters(), lr=1e-3)

Training loop with early stopping, as in the A1 search: stop once dev loss hasn't improved for 5 epochs, then go back to the best epoch's weights.

In [5]:
import copy
import time
from torch.utils.data import TensorDataset, DataLoader

loader = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)), batch_size=64, shuffle=True)
X_dev_t, y_dev_t = torch.tensor(X_dev).to(device), torch.tensor(y_dev).to(device)

PATIENCE = 5
best = {"dev_loss": float("inf"), "epoch": 0, "state": None}
start = time.perf_counter()

for epoch in range(1, 101):
    mlp.train()  # dropout on
    train_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(mlp(xb), yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(xb)

    mlp.eval()  # dropout off
    with torch.no_grad():
        dev_logits = mlp(X_dev_t)
        dev_loss = criterion(dev_logits, y_dev_t).item()
        dev_acc = (dev_logits.argmax(dim=1) == y_dev_t).float().mean().item()
    print(f"epoch {epoch:3d}  train loss {train_loss / len(X_tr):.3f}  dev loss {dev_loss:.3f}  dev acc {dev_acc*100:.2f}")

    if dev_loss < best["dev_loss"]:
        best = {"dev_loss": dev_loss, "epoch": epoch, "state": copy.deepcopy(mlp.state_dict())}
    elif epoch - best["epoch"] >= PATIENCE:
        break

mlp.load_state_dict(best["state"])
print(f"Training took {time.perf_counter() - start:.1f} s. Using epoch {best['epoch']} (lowest dev loss)")

epoch   1  train loss 0.666  dev loss 0.630  dev acc 51.69


epoch   2  train loss 0.464  dev loss 0.323  dev acc 87.29


epoch   3  train loss 0.281  dev loss 0.314  dev acc 86.44
epoch   4  train loss 0.236  dev loss 0.195  dev acc 94.07


epoch   5  train loss 0.187  dev loss 0.177  dev acc 93.22
epoch   6  train loss 0.165  dev loss 0.191  dev acc 94.07
epoch   7  train loss 0.154  dev loss 0.178  dev acc 93.22


epoch   8  train loss 0.166  dev loss 0.175  dev acc 93.22
epoch   9  train loss 0.152  dev loss 0.174  dev acc 89.83


epoch  10  train loss 0.137  dev loss 0.166  dev acc 92.37
epoch  11  train loss 0.125  dev loss 0.180  dev acc 93.22
epoch  12  train loss 0.118  dev loss 0.161  dev acc 92.37


epoch  13  train loss 0.142  dev loss 0.262  dev acc 87.29
epoch  14  train loss 0.146  dev loss 0.267  dev acc 87.29


epoch  15  train loss 0.146  dev loss 0.238  dev acc 89.83
epoch  16  train loss 0.108  dev loss 0.165  dev acc 93.22
epoch  17  train loss 0.105  dev loss 0.200  dev acc 88.98
Training took 2.7 s. Using epoch 12 (lowest dev loss)


## LR vs. MLP on the test set

In [6]:
mlp.eval()
with torch.no_grad():
    mlp_pred = mlp(torch.tensor(X_test).to(device)).argmax(dim=1).cpu().numpy()
mlp_acc = metrics.accuracy_score(y_test, mlp_pred)

print(f"LR  test accuracy: {lr_acc*100:.2f}")
print(f"MLP test accuracy: {mlp_acc*100:.2f}")
print(f"Difference: {(mlp_acc - lr_acc)*100:+.2f} points (1 point = {len(y_test) / 100:.0f} test messages)")

LR  test accuracy: 91.38
MLP test accuracy: 90.99
Difference: -0.39 points (1 point = 8 test messages)


This is one MLP with one seed, and the test set has only 777 messages (1 point ≈ 8 messages). A gap of a point or two can come from chance alone, so read it as "about the same". The dev set (10% of train) is also small, about 120 messages, so early stopping is noisier here.

# Mean pooling + tuned classifiers

Added after the hyperparameter search on mean-pooled features (`hparam_search_mean/`). Two things change from the sections above:
1. **Mean pooling:** each message's vector is the average of all its token vectors, instead of only `<s>`. ModernBERT's own classification head pools this way, and a quick check showed it helps ModernBERT a lot.
2. **Tuned settings**, chosen by 5-fold cross-validation on train: LR's `C` on this task's train set, the MLP's settings by the search on full 20 Newsgroups (the binary task wasn't searched). Nothing was chosen by looking at the test set.

Same data and 90/10 split as above, so the numbers are comparable.

In [7]:
CACHE_MEAN = "roberta_mean_sports.npz"

## Same as roberta_embed above, except that it averages the vectors of all of a message's tokens (mean pooling)
## instead of keeping only <s>. Padding is left out of the average.
def roberta_embed_mean(X, tokenizer, roberta, batch_size=32):
    vecs = []
    for i in tqdm(range(0, len(X), batch_size)):
        batch = tokenizer(X[i:i + batch_size], padding=True, truncation=True, max_length=512,
                          return_tensors="pt").to(device)
        with torch.no_grad():  # frozen: no gradients needed
            out = roberta(**batch)
        mask = batch["attention_mask"].unsqueeze(-1)  # (batch, tokens, 1): 1 = real token, 0 = padding
        vecs.append(((out.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)).cpu().numpy())
    return np.concatenate(vecs)

if os.path.exists(CACHE_MEAN):
    saved = np.load(CACHE_MEAN)
    X_train_mean, X_test_mean = saved["X_train"], saved["X_test"]
else:
    tokenizer = AutoTokenizer.from_pretrained("roberta-base")
    roberta = AutoModel.from_pretrained("roberta-base", add_pooling_layer=False).to(device).eval()
    X_train_mean = roberta_embed_mean(newsgroups_train.data, tokenizer, roberta)
    X_test_mean = roberta_embed_mean(newsgroups_test.data, tokenizer, roberta)
    np.savez(CACHE_MEAN, X_train=X_train_mean, X_test=X_test_mean)

print(X_train_mean.shape, X_test_mean.shape)

(1171, 768) (777, 768)


## Logistic regression, tuned

Standardized inputs and `C = 0.01`: the best of a `C` grid (1e-4 to 1e3) with and without standardizing, by 5-fold CV on this task's train set (90.35 CV accuracy). (On full 20 Newsgroups the best was standardized, `C = 0.0056`.) The `StandardScaler` is fit on the 90% only; the MLP below uses the same standardized inputs.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

X_tr_mean, X_dev_mean, y_tr, y_dev = train_test_split(
    X_train_mean, y_train, test_size=0.1, stratify=y_train, random_state=423)

scaler_mean = StandardScaler().fit(X_tr_mean)  # mean and std of the 90% only, so dev and test stay unseen
X_tr_mean, X_dev_mean = scaler_mean.transform(X_tr_mean), scaler_mean.transform(X_dev_mean)

with threadpool_limits(1):  # 1 BLAS thread: much faster on this machine
    logreg_mean = LogisticRegression(C=0.01, max_iter=5000).fit(X_tr_mean, y_tr)
lr_mean_acc = metrics.accuracy_score(y_test, logreg_mean.predict(scaler_mean.transform(X_test_mean)))
print(f"LR test accuracy: {lr_mean_acc*100:.2f}")

LR test accuracy: 89.83


## MLP, tuned

768 → 2048 → 32 → 2: the config with the best 3-seed CV accuracy in the search (72.42, on full 20 Newsgroups).
- **Dropout:** 0.2 on the input, 0.6 after hidden layer 1, 0.2 after hidden layer 2.
- **AdamW** at lr 3e-5, weight decay 0.1; **label smoothing** 0.2.
- GELU, batch size 32, standardized inputs, early stopping on dev loss with patience 10 (up to 300 epochs), as in the search.

In [9]:
torch.manual_seed(423)

mlp_mean = nn.Sequential(
    nn.Dropout(0.2),                                   # dropout on the input embedding
    nn.Linear(768, 2048), nn.GELU(), nn.Dropout(0.6),  # hidden layer 1
    nn.Linear(2048, 32), nn.GELU(), nn.Dropout(0.2),   # hidden layer 2
    nn.Linear(32, n_classes),                          # one logit per class
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.2)
optimizer = torch.optim.AdamW(mlp_mean.parameters(), lr=3e-5, weight_decay=0.1)

Same training loop as the search: early stopping on plain cross-entropy dev loss, patience 10, up to 300 epochs.

In [10]:
import copy
import time
from torch.utils.data import TensorDataset, DataLoader

loader = DataLoader(TensorDataset(torch.tensor(X_tr_mean), torch.tensor(y_tr)), batch_size=32, shuffle=True)
X_dev_t, y_dev_t = torch.tensor(X_dev_mean).to(device), torch.tensor(y_dev).to(device)

PATIENCE = 10
best = {"dev_loss": float("inf"), "epoch": 0, "state": None}
start = time.perf_counter()

for epoch in range(1, 301):
    mlp_mean.train()  # dropout on
    train_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(mlp_mean(xb), yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(xb)

    mlp_mean.eval()  # dropout off
    with torch.no_grad():
        dev_logits = mlp_mean(X_dev_t)
        dev_loss = nn.functional.cross_entropy(dev_logits, y_dev_t).item()  # no label smoothing, as in the search
        dev_acc = (dev_logits.argmax(dim=1) == y_dev_t).float().mean().item()
    print(f"epoch {epoch:3d}  train loss {train_loss / len(X_tr_mean):.3f}  dev loss {dev_loss:.3f}  dev acc {dev_acc*100:.2f}")

    if dev_loss < best["dev_loss"]:
        best = {"dev_loss": dev_loss, "epoch": epoch, "state": copy.deepcopy(mlp_mean.state_dict())}
    elif epoch - best["epoch"] >= PATIENCE:
        break

mlp_mean.load_state_dict(best["state"])
print(f"Training took {time.perf_counter() - start:.1f} s. Using epoch {best['epoch']} (lowest dev loss)")

epoch   1  train loss 0.673  dev loss 0.643  dev acc 77.97


epoch   2  train loss 0.628  dev loss 0.591  dev acc 84.75
epoch   3  train loss 0.587  dev loss 0.534  dev acc 86.44


epoch   4  train loss 0.540  dev loss 0.482  dev acc 88.14


epoch   5  train loss 0.512  dev loss 0.437  dev acc 88.98
epoch   6  train loss 0.485  dev loss 0.402  dev acc 89.83
epoch   7  train loss 0.464  dev loss 0.371  dev acc 89.83


epoch   8  train loss 0.458  dev loss 0.350  dev acc 91.53
epoch   9  train loss 0.449  dev loss 0.338  dev acc 89.83
epoch  10  train loss 0.439  dev loss 0.324  dev acc 91.53


epoch  11  train loss 0.438  dev loss 0.317  dev acc 89.83
epoch  12  train loss 0.435  dev loss 0.316  dev acc 90.68
epoch  13  train loss 0.424  dev loss 0.304  dev acc 92.37


... (18 outputs left out)

epoch  47  train loss 0.373  dev loss 0.269  dev acc 92.37


epoch  48  train loss 0.372  dev loss 0.271  dev acc 91.53


epoch  49  train loss 0.373  dev loss 0.274  dev acc 91.53


epoch  50  train loss 0.369  dev loss 0.275  dev acc 91.53
Training took 9.8 s. Using epoch 40 (lowest dev loss)


## `<s>` vs. mean pooling on the test set

In [11]:
mlp_mean.eval()
with torch.no_grad():
    mlp_mean_pred = mlp_mean(torch.tensor(scaler_mean.transform(X_test_mean)).to(device)).argmax(dim=1).cpu().numpy()
mlp_mean_acc = metrics.accuracy_score(y_test, mlp_mean_pred)

rows = [("<s>, first-notebook settings (above)", lr_acc, mlp_acc),
        ("mean pooling, tuned settings", lr_mean_acc, mlp_mean_acc)]
print(f"{'test accuracy':<42}{'LR':>7}{'MLP':>8}")
for name, lr_a, mlp_a in rows:
    print(f"{name:<42}{lr_a*100:>7.2f}{mlp_a*100:>8.2f}")
print(f"(1 point = {len(y_test) / 100:.0f} test messages)")

test accuracy                                  LR     MLP
<s>, first-notebook settings (above)        91.38   90.99
mean pooling, tuned settings                89.83   91.38
(1 point = 8 test messages)


This is one MLP with one seed, and the test set has only 777 messages (1 point ≈ 8 messages). A gap of a point or two can come from chance alone, so only larger gaps mean much. The settings were searched on full 20 Newsgroups, not on this task.